# vLLM: offline inference (Colab)

Этот ноутбук показывает **offline inference** через vLLM в Google Colab: загрузка маленькой модели и генерация без запуска HTTP-сервера.

> ⚠️ **Нужна GPU:** в Colab включите Runtime → Change runtime type → GPU (T4 или лучше). Без GPU vLLM не запустится.

Модель `facebook/opt-125m` — очень маленькая, помещается на одной GPU и подходит для проверки установки и API.

## 1. Установка vLLM

В Colab обычно достаточно `pip install vllm`. При первом запуске установка может занять 1–2 минуты.

In [ ]:
!pip install -q vllm
print('vLLM installed')

## 2. Проверка GPU

vLLM требует CUDA. Убедимся, что GPU виден.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

## 3. Загрузка модели и генерация

Используем модель **facebook/opt-125m** — небольшая, быстро загружается. Для чат-моделей (TinyLlama, Llama) можно заменить на `TinyLlama/TinyLlama-1.1B-Chat-v1.0` и формировать промпты в формате чата.

In [ ]:
from vllm import LLM, SamplingParams

llm = LLM(model="facebook/opt-125m", trust_remote_code=True)

sampling_params = SamplingParams(
    temperature=0.8,
    top_p=0.95,
    max_tokens=64,
)

prompts = [
    "The capital of France is",
    "One plus one equals",
]
outputs = llm.generate(prompts, sampling_params)

for i, out in enumerate(outputs):
    print(f"Prompt: {prompts[i]}")
    print(f"Output: {out.outputs[0].text}")
    print("---")

## 4. (Опционально) Чат-модель TinyLlama

Если хотите попробовать чат-модель, раскомментируйте ячейку ниже. TinyLlama занимает больше памяти; на T4 должно поместиться. Промпт собирается в формате, который ожидает модель.

In [ ]:
# from vllm import LLM, SamplingParams

# llm = LLM(model="TinyLlama/TinyLlama-1.1B-Chat-v1.0", trust_remote_code=True)
# sampling = SamplingParams(temperature=0.7, max_tokens=128)

# Формат промпта для TinyLlama Chat
# prompt = "<|system|>\nYou are a helpful assistant.\n<|user|>\nWhat is RAG?\n<|assistant|>\n"
# outputs = llm.generate([prompt], sampling)
# print(outputs[0].outputs[0].text)

## Дальше

- **Сервер:** на своей машине с GPU запустите `vllm serve <model>` и подключайтесь к нему через OpenAI-клиент с `base_url="http://<host>:8000/v1"`.
- **Документация:** [vLLM Quickstart](https://docs.vllm.ai/en/stable/getting_started/quickstart.html), [Offline Inference](https://docs.vllm.ai/en/stable/examples/offline_inference/basic.html).
- **В хранилище:** [[13_module_vllm/README|Модуль 13 — vLLM]], [[13_module_vllm/01_vllm_serve_example|Пример: сервер и offline inference]].